# 17. Generators & Iterators (5+ Years Interview Guide)
Deep architectural analysis of the iterator protocol, generator state machines, yield vs return, coroutine send/throw/close protocols, yield from delegation, and memory benchmarking.

### Key 5-Year Interview Concepts Covered:
- **Iterator Protocol Mechanics**: Implementing `__iter__()` and `__next__()` with `StopIteration`.
- **Generator Frame Preservation**: How CPython pauses execution frames (`PyGenObject`) on heap memory with O(1) memory usage.
- **Subgenerator Delegation (`yield from`)**: Transparent bidirectional communication and value passing.
- **Coroutine-Style Generators**: Bi-directional data pipelines with `.send()`, `.throw()`, and `.close()`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. The Iterator Protocol (`__iter__` & `__next__`)
**Explanation**: An iterator is an object representing a stream of data. It must implement two methods: `__iter__()` (which returns the iterator object itself) and `__next__()` (which returns the next item in sequence, or raises `StopIteration` when elements are exhausted).

**Syntax**: `def __iter__(self): return self` / `def __next__(self): ...`

In [2]:
transaction_iterator = iter([1])
print(next(transaction_iterator))

1


### 2. Generator Functions & The `yield` Keyword
**Explanation**: Any function containing the `yield` keyword is a generator function. Calling a generator function does NOT execute its body immediately; instead, it returns a generator object (`PyGenObject`). Each time `next(gen)` is called, the function executes until it reaches a `yield`, yields the value, and suspends its execution state.

**Syntax**: `def generate_numbers(): yield 1; yield 2`

In [3]:
def suspended_frame_generator(): yield 1
print(next(suspended_frame_generator()))

1


### 3. Generator Execution Frame Preservation
**Explanation**: When a standard function returns, its call stack frame is destroyed. When a generator yields, CPython freezes the frame object (`f_back`, `f_locals`, instruction pointer `f_lasti`) on the heap. Calling `next()` resumes execution immediately after the yield point with all local variables preserved.

**Syntax**: `gen = generator_fn(); item = next(gen)`

In [4]:
def stateful_generator():
    running_state = 10
    yield running_state
    running_state += 10
    yield running_state
generator_instance = stateful_generator()
next(generator_instance)
print('Suspended state:', next(generator_instance))

Suspended state: 20


### 4. Generator Expressions (Lazy Comprehensions)
**Explanation**: Generator expressions use parentheses `(expr for item in iterable if cond)`. Unlike list comprehensions which build the entire list in RAM immediately, generator expressions produce items on-demand with O(1) auxiliary memory footprint.

**Syntax**: `lazy_gen = (x * 2 for x in large_sequence)`

In [5]:
generator_expression_object = (x for x in range(3))
print(next(generator_expression_object))

0


### 5. Memory Footprint Comparison: List vs Generator
**Explanation**: A list holding 1,000,000 integers consumes ~8 MB of RAM, while a generator producing 1,000,000 integers consumes only ~120 bytes regardless of stream length. In enterprise data engineering, always use generators to stream multi-gigabyte files or database query cursors.

**Syntax**: `sys.getsizeof(gen_expr) << sys.getsizeof(list_comp)`

In [6]:
import sys
payout_list = [x for x in range(1000)]
generator_expression_object = (x for x in range(1000))
print('List:', sys.getsizeof(payout_list), 'bytes | Gen:', sys.getsizeof(generator_expression_object), 'bytes')

List: 8856 bytes | Gen: 192 bytes


### 6. Coroutine Input Streaming (`generator.send()`)
**Explanation**: `yield` is an expression that can receive values sent by the caller: `received_val = yield output_val`. Callers inject values using `gen.send(value)`. Note: A newly created generator must be 'primed' by calling `next(gen)` or `gen.send(None)` before sending non-None values.

**Syntax**: `val = yield output; gen.send(input_val)`

In [7]:
def coroutine_function():
    input_value = yield 'Started'
    yield f'Received: {input_value}'
coroutine_instance = coroutine_function()
print(next(coroutine_instance))
print(coroutine_instance.send(100))

Started
Received: 100


### 7. Injecting Exceptions (`generator.throw()`)
**Explanation**: `gen.throw(ExceptionType, value)` raises an exception inside the generator frame at the suspended `yield` statement. The generator can catch the exception using `try-except` to perform recovery or cleanup logic.

**Syntax**: `gen.throw(ValueError, 'Invalid transaction payload')`

In [8]:
def error_yielding_generator():
    try: yield 1
    except ValueError: yield 'Caught'
generator_instance = error_yielding_generator()
next(generator_instance)
print(generator_instance.throw(ValueError))

Caught


### 8. Terminating Generators Cleanly (`generator.close()`)
**Explanation**: `gen.close()` raises a `GeneratorExit` exception inside the suspended generator frame. Standard `finally` blocks execute to ensure file handles, database cursors, and network sockets are cleanly closed.

**Syntax**: `gen.close()`

In [9]:
def cleanup_generator():
    try: yield 1
    finally: print('Closed')
generator_instance = cleanup_generator()
next(generator_instance)
generator_instance.close()

Closed


### 9. Subgenerator Delegation (`yield from`)
**Explanation**: Introduced in PEP 380, `yield from subgenerator` delegates iteration to a subgenerator. It transparently forwards yields, handles `send()` and `throw()` bidirectional communication, and captures the subgenerator's `return` value.

**Syntax**: `yield from subgenerator_or_iterable`

In [10]:
def sub_generator(): yield 1
def parent_generator(): yield from sub_generator()
print(list(parent_generator()))

[1]


### 10. Streaming Infinite Sequences
**Explanation**: Because generators compute values on-demand, they can represent infinite streams (e.g. continuous timestamp tickers, UUID generators, Fibonacci sequence) without running out of memory.

**Syntax**: `def infinite_ticker(): count = 0; while True: yield count; count += 1`

In [11]:
def infinite_counter_generator():
    counter = 0
    while True: yield counter; counter += 1
generator_instance = infinite_counter_generator()
print(next(generator_instance), next(generator_instance))

0 1


### 11. Infinite Generators with Dynamic Exit Conditions
**Explanation**: Infinite generators can be consumed safely using `itertools.islice()` or explicit `break` conditions in caller loops, allowing flexible data processing pipelines.

**Syntax**: `from itertools import islice; first_ten = list(islice(infinite_gen, 10))`

In [12]:
def self_limiting_generator():
    yield 1
    return
print(list(self_limiting_generator()))

[1]


### 12. Reusable Iterator Class Wrappers
**Explanation**: Generators are one-shot iterables: once exhausted, they cannot be restarted. To create a reusable iterable that can be looped over multiple times (e.g. in multiple `for` loops), implement a class with `def __iter__(self):` that yields from a fresh generator on every call.

**Syntax**: `class ReusableStream: def __iter__(self): with open(self.path) as f: yield from f`

In [13]:
class ReusableGenerator:
    def __iter__(self):
        yield 1
        yield 2
reusable_instance = ReusableGenerator()
print(list(reusable_instance), list(reusable_instance))

[1, 2] [1, 2]


### 13. Inspecting Generator Lifecycle States (`inspect` module)
**Explanation**: The `inspect` module provides `getgeneratorstate(gen)`, returning one of 4 lifecycle states: `GEN_CREATED` (not yet started), `GEN_RUNNING` (currently executing), `GEN_SUSPENDED` (waiting at yield), or `GEN_CLOSED` (exhausted or closed).

**Syntax**: `import inspect; state = inspect.getgeneratorstate(gen)`

In [14]:
import inspect
def dummy_generator(): yield 1
generator_instance = dummy_generator()
print('State:', inspect.getgeneratorstate(generator_instance))
next(generator_instance)
print('State after next:', inspect.getgeneratorstate(generator_instance))


State: GEN_CREATED
State after next: GEN_SUSPENDED


### 14. Recursive Generator Flattening
**Explanation**: Using `yield from` with recursive generators elegantly flattens deeply nested, arbitrarily structured trees or JSON hierarchies in O(depth) stack space.

**Syntax**: `def flatten(nested): for item in nested: if isinstance(item, list): yield from flatten(item) else: yield item`

In [15]:
def flatten_nested_lists(nested_list):
    for item in nested_list:
        if isinstance(item, list): yield from flatten_nested_lists(item)
        else: yield item
print(list(flatten_nested_lists([1, [2, 3]])))

[1, 2, 3]


### 15. Composable Generator Pipelines
**Explanation**: Connecting generators in a pipeline `clean_gen = filter_tx(parse_tx(read_file(path)))` streams data end-to-end: each row is read, parsed, filtered, and aggregated one at a time, keeping RAM consumption constant across billions of records.

**Syntax**: `pipeline = summarize(filter_records(parse_lines(file_stream)))`

In [16]:
generator_stage_one = (x for x in range(5))
generator_stage_two = (x * 2 for x in generator_stage_one if x % 2 == 0)
print('Chained pipeline output:', list(generator_stage_two))


Chained pipeline output: [0, 4, 8]


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Constant-memory CSV streaming, lazy fraud threshold detection, and generator pipeline orchestration across financial transaction logs.


In [17]:
# Solution:
def read_amts(path):
    with open(path, 'r') as f:
        f.readline()
        for _ in range(100):
            row = f.readline().strip().split(',')
            yield float(row[3]) if row[3] not in ('', 'NaN') else 0.0

amts = read_amts(csv_path)
filtered_amts = (a for a in amts if a >= 100.0)
taxed_amts = (round(a * 0.18, 2) for a in filtered_amts)
print('Sample taxed amounts:', [next(taxed_amts) for _ in range(5)])


Sample taxed amounts: [218.94, 58.5, 24.6, 22.36, 231.24]


### Q2: Dynamic Generator State Lifecycle Tracking
**Explanation**: **Scenario**: Implement a transaction batch generator and inspect its `getgeneratorstate()` transitions (`GEN_CREATED` -> `GEN_SUSPENDED` -> `GEN_CLOSED`) during execution.

**Syntax**: `inspect.getgeneratorstate(batch_generator)`

In [18]:
# Solution:
import inspect
def audit_gen():
    yield 'CHECK_1'
    yield 'CHECK_2'

ag = audit_gen()
print('State before execution:', inspect.getgeneratorstate(ag))
next(ag)
print('State during execution:', inspect.getgeneratorstate(ag))


State before execution: GEN_CREATED
State during execution: GEN_SUSPENDED
